In [29]:
import json
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
import os


# LLM

In [30]:
load_dotenv(override=True)

True

In [31]:
from langchain_ollama import ChatOllama
llm=ChatOllama(model="nemotron-3-super:cloud", temperature=0)

# Implementation de RAG (avec nc346.pdf)

In [32]:
pdf_file="./pdfs/nc346.pdf"

In [33]:
pdf_loader=PyPDFLoader(pdf_file)

In [34]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

In [35]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)

161

# Vectore Store - ChromaDB et Embedding

In [36]:
from langchain_ollama import OllamaEmbeddings
embedding_model=OllamaEmbeddings(model="nomic-embed-text")

In [37]:
vectorstore=Chroma.from_documents(chunks,embedding_model,collection_name="nc346_v2",persist_directory="./store")

**Trouver les chunks liée à la questions (Retriever)**

In [38]:
# la création Retriever
retriever=vectorstore.as_retriever(search_type='similarity',search_kwargs={'k':5})
# k=5 c-à-d les cinq plus proches chunks à la question posée

In [40]:
# on fait un test ici
retrieved_chunks = retriever.invoke("QU'il est la capitalisation boursière de Maroc pour 2025 ?")
print(retrieved_chunks)
print("nombre de chunks est :", len(retrieved_chunks))


[Document(metadata={'source': './pdfs/nc346.pdf', 'producer': 'Adobe PDF Library 17.0', 'trapped': '/False', 'creationdate': '2025-12-26T11:38:00+01:00', 'page_label': '21', 'moddate': '2025-12-26T11:38:19+01:00', 'creator': 'Adobe InDesign 20.5 (Macintosh)', 'page': 20, 'total_pages': 52}, page_content='En 2026, la croissance de la demande mondiale resterait proche de 2025, mais à un rythme \nlégèrement inférieur au trend : l’AIE s’attend à +0,9 mbj, freinée par une conjoncture moins \nporteuse, la substitution (électricité/gaz/solaire) et l’électrification, tandis que les charges \npétrochimiques deviendraient le principal moteur. Côté offre, l’augmentation resterait nettement \nplus forte (+2,4 mbj en 2026), tirée à la fois par les producteurs non-OPEP+ (+1,2 mbj), notamment \ndes Amériques, et par l’OPEP+ (+1,2 mbj) malgré les contraintes liées aux sanctions. La balance \nresterait largement excédentaire. L ’AIE prévoit un surplus moyen d’environ 3,8 mbj en 2026, avec \npoursuite d

# RAG et Q&A

In [41]:
# Prompt Design
prompt_template="""
Answer the following question based only on provided context The context is about NOTE DE
CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

*Retrieving the Relevant Documents*

In [63]:
user_input = "QU'il est la capitalisation boursière de Maroc pour 2025 ?"

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)


context_for_query


'En 2026, la croissance de la demande mondiale resterait proche de 2025, mais à un rythme \nlégèrement inférieur au trend : l’AIE s’attend à +0,9 mbj, freinée par une conjoncture moins \nporteuse, la substitution (électricité/gaz/solaire) et l’électrification, tandis que les charges \npétrochimiques deviendraient le principal moteur. Côté offre, l’augmentation resterait nettement \nplus forte (+2,4 mbj en 2026), tirée à la fois par les producteurs non-OPEP+ (+1,2 mbj), notamment \ndes Amériques, et par l’OPEP+ (+1,2 mbj) malgré les contraintes liées aux sanctions. La balance \nresterait largement excédentaire. L ’AIE prévoit un surplus moyen d’environ 3,8 mbj en 2026, avec \npoursuite des reconstructions de stocks 16.\nDans ce contexte, le biais sur les prix pétroliers reste baissier à moyen terme, la détente reflétant \navant tout l’excès d’offre. Le Brent pourrait ainsi glisser vers 60 $/b en 2026, après une moyenne. En 2026, la croissance de la demande mondiale resterait proche de 2

In [60]:
len(relevant_document_chunks)

5

In [61]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)


Answer the following question based only on provided context The context is about NOTE DE
CONJONCTURE DIRECTION DES ETUDES ET DES PREVISIONS FINANCIERES Décembre 2025
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
8
Note de Conjoncture 
L ’économie de la zone euro montre une croissance modérée, mais globalement résiliente. Selon 
les dernières prévisions de l’OCDE (décembre 2025), le PIB progresserait de 1,3% en 2025 après 
0,8% en 2024. Il marquerait ensuite un léger fléchissement à 1,2% en 2026, avant de réaccélérer 
à 1,4% en 2027, à mesure que la demande intérieure se raffermit et que le commerce extérieur se 
redresse.
Cette trajectoire s’appuie sur la résilience du marché du travail, la désinflation et la hausse 
des revenus réels, qui soutiennent la consommation. L ’investissement privé reste freiné par 
l’incertitude, malgré l’amélioration des conditions de 

*récupérer la réponse de la part de llm*

In [ ]:
resp=llm.invoke(prompt)
from IPython.display import Markdown
print(display(Markdown(resp.content)))

Je ne sais pas!

None


*Definire rag comme fonction*

In [50]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)   
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content


In [51]:
resp=RAG(" qu'ils sont les événements culturels dans le sud-est de Maroc ? ")
print(display(Markdown(resp)))

Je ne sais pas!

None


# Evaluation

In [52]:
#question ou query 
user_input=" QU'il est la capitalisation boursière de Maroc pour 2025 "

In [53]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [54]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [55]:
# Réponse d'un RAG
answer=RAG(" QU'il est la capitalisation boursière de Maroc pour 2025 ? ")
print(display(Markdown(answer)))

Je ne sais pas!

None


Le prompt pour le juge

In [56]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [57]:
# définir le juge
groundness_checker=ChatOllama(
      model="gpt-oss:120b-cloud",
      temperature=0
)

In [58]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question) 
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response = model.invoke(prompt)
    return juge_response.content


In [ ]:
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
print(display(Markdown(resp)))

**Étapes d’évaluation de la conformité à la métrique**

1. **Identifier la demande de l’utilisateur**  
   - Lire la question et repérer l’information recherchée (ici : « la capitalisation boursière du Maroc pour 2025 »).

2. **Analyser le contexte fourni**  
   - Parcourir le texte du contexte et relever toutes les données susceptibles de répondre à la question.  
   - Noter s’il existe aucune mention du Maroc, de sa capitalisation boursière, ou de l’année 2025.

3. **Comparer le contenu de la réponse avec le contexte**  
   - Vérifier que chaque affirmation de la réponse se trouve explicitement dans le contexte.  
   - S’assurer qu’aucune information extérieure (connaissances générales, hypothèses, inventions) n’est introduite.

4. **Évaluer la façon dont la réponse gère l’absence d’information**  
   - Si le contexte ne contient pas la donnée recherchée, une réponse admissible peut indiquer « information non disponible » ou « je ne sais pas », tant que cela ne prétend pas ajouter de nouvelles données.

5. **Attribuer un score selon la grille de notation**  
   - 1 : aucune conformité.  
   - 2 : conformité très limitée.  
   - 3 : conformité correcte mais imparfaite.  
   - 4 : bonne conformité, presque totale.  
   - 5 : conformité totale à la métrique.

---

**Application à l’exemple**

1. **Question** : « Quelle est la capitalisation boursière du Maroc pour 2025 ? »

2. **Contexte** : Le texte porte exclusivement sur les prévisions de l’AIE concernant la demande et l’offre pétrolières en 2026 (et références à 2025 uniquement dans le cadre de la comparaison de la demande). Aucun passage ne mentionne le Maroc, ni sa capitalisation boursière, ni l’année 2025 sous cet aspect.

3. **Réponse IA** : « Je ne sais pas ! »

   - La réponse ne fournit aucune donnée qui ne soit pas dans le contexte.  
   - Elle indique simplement l’absence d’information, ce qui est une conclusion dérivée du fait que le contexte ne contient pas la donnée recherchée.

4. **Conformité**  
   - La réponse respecte strictement la métrique : elle ne crée ni n’invente aucune information extérieure.  
   - Elle se contente de refléter le manque de donnée disponible dans le contexte.

5. **Score**  
   - La métrique est **entièrement respectée**.  
   - Score : **5**.

None
